# Create Bronze Tables - Initialize Bronze Layer Schema

This notebook creates the `bronze` database and all bronze layer tables with raw data schemas.
The bronze layer stores data exactly as received from source systems with audit columns.

**Tables created (in `unity_catalog.bronze`):**
- `transactions_raw`
- `transaction_items_raw`
- `subscriptions_raw`
- `customer_interactions_raw`
- `product_catalog_raw`
- `inventory_snapshots_raw`
- `marketing_campaigns_raw`
- `campaign_events_raw`

> In Databricks the `spark` session is pre-defined, so there is no need to build or stop a `SparkSession`.

In [ ]:
from datetime import datetime

print("=" * 80)
print("  Create Bronze Tables")
print("=" * 80)
print(f"\nStarting table creation at: {datetime.now()}")

## 1. Create bronze database

In [ ]:
spark.sql("CREATE DATABASE IF NOT EXISTS unity_catalog.bronze")
print("  ✓ Bronze database ready")

## 2. Define bronze table DDLs

In [ ]:
tables = {
    "transactions_raw": """
        CREATE TABLE IF NOT EXISTS unity_catalog.bronze.transactions_raw
        (
            transaction_id STRING,
            customer_id STRING,
            transaction_timestamp STRING,
            channel STRING,
            store_id STRING,
            payment_method STRING,
            payment_status STRING,
            subtotal STRING,
            tax_amount STRING,
            shipping_cost STRING,
            discount_amount STRING,
            total_amount STRING,
            currency STRING,
            loyalty_points_earned STRING,
            loyalty_points_redeemed STRING,
            coupon_codes STRING,
            _source_system STRING,
            _ingestion_timestamp TIMESTAMP,
            _file_name STRING,
            _record_offset BIGINT
        )
        PARTITIONED BY (ingestion_date DATE)
    """,
    "transaction_items_raw": """
        CREATE TABLE IF NOT EXISTS unity_catalog.bronze.transaction_items_raw
        (
            transaction_item_id STRING,
            transaction_id STRING,
            product_id STRING,
            variant_id STRING,
            quantity STRING,
            unit_price STRING,
            discount_percentage STRING,
            tax_rate STRING,
            return_quantity STRING,
            return_reason STRING,
            fulfillment_status STRING,
            warehouse_id STRING,
            _source_system STRING,
            _ingestion_timestamp TIMESTAMP,
            _file_name STRING,
            _record_offset BIGINT
        )
        PARTITIONED BY (ingestion_date DATE)
    """,
    "subscriptions_raw": """
        CREATE TABLE IF NOT EXISTS unity_catalog.bronze.subscriptions_raw
        (
            subscription_id STRING,
            customer_id STRING,
            subscription_type STRING,
            plan_id STRING,
            start_date STRING,
            end_date STRING,
            status STRING,
            billing_frequency STRING,
            subscription_amount STRING,
            next_billing_date STRING,
            auto_renewal STRING,
            cancellation_date STRING,
            cancellation_reason STRING,
            _source_system STRING,
            _ingestion_timestamp TIMESTAMP,
            _file_name STRING,
            _record_offset BIGINT
        )
        PARTITIONED BY (ingestion_date DATE)
    """,
    "customer_interactions_raw": """
        CREATE TABLE IF NOT EXISTS unity_catalog.bronze.customer_interactions_raw
        (
            interaction_id STRING,
            customer_id STRING,
            interaction_timestamp STRING,
            interaction_type STRING,
            channel STRING,
            agent_id STRING,
            category STRING,
            subcategory STRING,
            sentiment_score STRING,
            resolution_time_minutes STRING,
            satisfaction_rating STRING,
            notes STRING,
            _source_system STRING,
            _ingestion_timestamp TIMESTAMP,
            _file_name STRING,
            _record_offset BIGINT
        )
        PARTITIONED BY (ingestion_date DATE)
    """,
    "product_catalog_raw": """
        CREATE TABLE IF NOT EXISTS unity_catalog.bronze.product_catalog_raw
        (
            product_id STRING,
            product_name STRING,
            category_level1 STRING,
            category_level2 STRING,
            category_level3 STRING,
            brand STRING,
            manufacturer STRING,
            unit_cost STRING,
            list_price STRING,
            margin_percentage STRING,
            supplier_id STRING,
            lead_time_days STRING,
            weight_kg STRING,
            dimensions STRING,
            tags STRING,
            launch_date STRING,
            discontinuation_date STRING,
            _source_system STRING,
            _ingestion_timestamp TIMESTAMP,
            _file_name STRING,
            _record_offset BIGINT
        )
        PARTITIONED BY (ingestion_date DATE)
    """,
    "inventory_snapshots_raw": """
        CREATE TABLE IF NOT EXISTS unity_catalog.bronze.inventory_snapshots_raw
        (
            snapshot_id STRING,
            snapshot_timestamp STRING,
            product_id STRING,
            variant_id STRING,
            warehouse_id STRING,
            quantity_on_hand STRING,
            quantity_reserved STRING,
            quantity_available STRING,
            reorder_point STRING,
            reorder_quantity STRING,
            _source_system STRING,
            _ingestion_timestamp TIMESTAMP,
            _file_name STRING,
            _record_offset BIGINT
        )
        PARTITIONED BY (ingestion_date DATE)
    """,
    "marketing_campaigns_raw": """
        CREATE TABLE IF NOT EXISTS unity_catalog.bronze.marketing_campaigns_raw
        (
            campaign_id STRING,
            campaign_name STRING,
            campaign_type STRING,
            channel STRING,
            start_date STRING,
            end_date STRING,
            budget STRING,
            target_audience STRING,
            creative_id STRING,
            _source_system STRING,
            _ingestion_timestamp TIMESTAMP,
            _file_name STRING,
            _record_offset BIGINT
        )
        PARTITIONED BY (ingestion_date DATE)
    """,
    "campaign_events_raw": """
        CREATE TABLE IF NOT EXISTS unity_catalog.bronze.campaign_events_raw
        (
            event_id STRING,
            campaign_id STRING,
            customer_id STRING,
            event_timestamp STRING,
            event_type STRING,
            device_type STRING,
            location STRING,
            attributed_revenue STRING,
            _source_system STRING,
            _ingestion_timestamp TIMESTAMP,
            _file_name STRING,
            _record_offset BIGINT
        )
        PARTITIONED BY (ingestion_date DATE)
    """
}

## 3. Create each bronze table

In [ ]:
table_list = list(tables.keys())
created_tables = []
failed_tables = []

for idx, (table_name, ddl) in enumerate(tables.items(), 1):
    print(f"[{idx}/{len(table_list)}] Creating table: unity_catalog.bronze.{table_name}")
    try:
        spark.sql(ddl)
        print(f"  ✓ Table created: unity_catalog.bronze.{table_name}")
        created_tables.append(table_name)
    except Exception as e:
        print(f"  ❌ Failed to create {table_name}: {str(e)}")
        failed_tables.append(table_name)

## 4. Bronze tables summary

In [ ]:
tables_df = spark.sql("SHOW TABLES IN unity_catalog.bronze")
table_count = tables_df.count()
print(f"Total tables in unity_catalog.bronze database: {table_count}\n")
tables_df.show(truncate=False)

In [ ]:
print("=" * 80)
if len(failed_tables) == 0:
    print("  SUCCESS! All bronze tables created")
    print("=" * 80)
    print(f"\nCreated {len(created_tables)}/{len(table_list)} tables:")
    for table in created_tables:
        print(f"  - unity_catalog.bronze.{table}")
    print("\nNext steps:")
    print("  1. Run ingest_bronze_data.ipynb to load data")
    print("  2. Run validate_bronze_data to validate")
else:
    print("  COMPLETED WITH ERRORS")
    print("=" * 80)
    print(f"\nCreated: {len(created_tables)}/{len(table_list)} tables")
    print(f"Failed: {len(failed_tables)}/{len(table_list)} tables")
    if failed_tables:
        print("\nFailed tables:")
        for table in failed_tables:
            print(f"  - unity_catalog.bronze.{table}")